<a href="https://colab.research.google.com/github/31Pritika/Mello/blob/main/Mello.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

ratings = pd.read_csv('/content/drive/MyDrive/moviesdataset/ratings_small.csv')
movies = pd.read_csv('/content/drive/MyDrive/moviesdataset/movies_metadata.csv')

/tmp/ipykernel_2332/669592158.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('/content/drive/MyDrive/moviesdataset/movies_metadata.csv')


In [ ]:
print(f"movies_metadata.csv: {movies.shape}")
print(f"rating_small.csv: {ratings.shape}")

movies_metadata.csv: (45466, 24)
rating_small.csv: (100004, 4)


In [ ]:
movies = movies[movies["id"].astype(str).str.isnumeric()]
movies["id"] = movies["id"].astype(int)

# Keep only the columns you actually need for now
movies_clean = movies[["id", "title", "genres", "overview", "vote_average", "vote_count"]].copy()
movies_clean = movies_clean.rename(columns={"id": "movieId_tmdb"})

In [ ]:
# ---------------------------------------------------------------
# 3. ratings.csv uses MovieLens movieId, NOT TMDB id — need links.csv to bridge them.
#    If you downloaded links.csv, use it. Otherwise this dataset's ratings_small.csv
#    already uses the same movieId space as movies_metadata's "id" column via links.
# ---------------------------------------------------------------
try:
    links = pd.read_csv("links.csv")
    links = links.dropna(subset=["tmdbId"])
    links["tmdbId"] = links["tmdbId"].astype(int)
    ratings = ratings.merge(links[["movieId", "tmdbId"]], on="movieId", how="inner")
    ratings = ratings.rename(columns={"tmdbId": "movieId_tmdb"})
    print("Linked ratings to TMDB ids via links.csv")
except FileNotFoundError:
    print("links.csv not found — skipping TMDB id bridge for now")
    ratings["movieId_tmdb"] = ratings["movieId"]  # fallback, fix later


links.csv not found — skipping TMDB id bridge for now


In [ ]:
# ---------------------------------------------------------------
# 4. Build a SPARSE user-item matrix (not a dense pivot table — this dataset is too big for that)
# ---------------------------------------------------------------
print("\nBuilding sparse user-item matrix...")

# Map raw userId / movieId_tmdb to dense 0..N-1 indices required by sparse matrices
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId_tmdb"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

row_idx = ratings["userId"].map(user_to_idx).values
col_idx = ratings["movieId_tmdb"].map(movie_to_idx).values
data = ratings["rating"].values

sparse_matrix = csr_matrix((data, (row_idx, col_idx)),
                            shape=(len(user_ids), len(movie_ids)))

print(f"Sparse matrix shape: {sparse_matrix.shape}")
print(f"Non-zero entries: {sparse_matrix.nnz}  "
      f"({100 * sparse_matrix.nnz / (sparse_matrix.shape[0]*sparse_matrix.shape[1]):.4f}% filled)")



Building sparse user-item matrix...
Sparse matrix shape: (671, 9066)
Non-zero entries: 100004  (1.6439% filled)


In [ ]:
# ---------------------------------------------------------------
# 5. Sanity check — look at one user's ratings joined back to real titles
# ---------------------------------------------------------------
sample_user = ratings["userId"].iloc[0]
sample_ratings = ratings[ratings["userId"] == sample_user].merge(
    movies_clean, on="movieId_tmdb", how="left"
)
print(f"\nSample: user {sample_user}'s rated movies")
print(sample_ratings[["title", "rating"]].head(10))



Sample: user 1's rated movies
  title  rating
0   NaN     2.5
1   NaN     3.0
2   NaN     3.0
3   NaN     2.0
4   NaN     4.0
5   NaN     2.0
6   NaN     2.0
7   NaN     2.0
8   NaN     3.5
9   NaN     2.0


In [ ]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=20)
user_factors = svd.fit_transform(sparse_matrix)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = cosine_similarity(user_factors)

In [ ]:
print("\nRunning SVD to compress users into taste patterns...")
n_components = 20  # how many "hidden taste dimensions" to keep — start small, tune later
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(sparse_matrix)

print(f"user_factors shape: {user_factors.shape}  "
      f"(each user is now a {n_components}-number taste fingerprint instead of a full row of ratings)")
print(f"Variance explained by these {n_components} components: "
      f"{svd.explained_variance_ratio_.sum():.2%}")


Running SVD to compress users into taste patterns...
user_factors shape: (671, 20)  (each user is now a 20-number taste fingerprint instead of a full row of ratings)
Variance explained by these 20 components: 37.54%


In [ ]:
# 7. Compare users to each other — this IS the match score
# ---------------------------------------------------------------
from sklearn.metrics.pairwise import cosine_similarity

print("\nComputing user-to-user similarity...")
similarity_matrix = cosine_similarity(user_factors)
print(f"similarity_matrix shape: {similarity_matrix.shape}  (every user compared to every other user)")

# Sanity check: find the most similar OTHER user to user index 0, and compare their tastes
idx_to_user = {v: k for k, v in user_to_idx.items()}
user0_similarities = similarity_matrix[0].copy()
user0_similarities[0] = -1  # exclude comparing user 0 to themselves
most_similar_idx = user0_similarities.argmax()
most_similar_userId = idx_to_user[most_similar_idx]

print(f"\nMost similar user to userId {idx_to_user[0]}: userId {most_similar_userId} "
      f"(similarity score: {user0_similarities[most_similar_idx]:.3f})")


Computing user-to-user similarity...
similarity_matrix shape: (671, 671)  (every user compared to every other user)

Most similar user to userId 1: userId 58 (similarity score: 0.797)


In [ ]:

# Show what that "most similar" person actually rated, so you can eyeball if it's a sensible match
similar_user_ratings = ratings[ratings["userId"] == most_similar_userId].merge(
    movies_clean, on="movieId_tmdb", how="left"
)
print(f"\nWhat userId {most_similar_userId} rated:")
print(similar_user_ratings[["title", "rating"]].head(10))


What userId 58 rated:
                  title  rating
0             Star Wars     5.0
1            Metropolis     1.0
2               48 Hrs.     5.0
3      Bang, Boom, Bang     1.0
4      Summer in Berlin     3.0
5        Batman & Robin     2.0
6      The Last Samurai     4.0
7                   NaN     4.0
8  Meet Me in St. Louis     2.0
9      Dawn of the Dead     5.0


In [ ]:
# ---------------------------------------------------------------
# 8. Evaluate properly: precision@k
#    Idea: hide some of each user's real ratings (test set), train only on the
#    rest (train set), then check whether the model's top-K guesses for that
#    user actually land on movies they genuinely rated highly.
# ---------------------------------------------------------------
from sklearn.model_selection import train_test_split

print("\nEvaluating with precision@k...")

LIKE_THRESHOLD = 4.0   # a rating >= this counts as "the user actually liked it"
K = 10                 # how many recommendations we check per user

train_ratings, test_ratings = train_test_split(
    ratings, test_size=0.2, random_state=42, stratify=None
)


Evaluating with precision@k...


In [ ]:
# Rebuild the sparse matrix using ONLY the training ratings —
# the model must never see test ratings during training, or the evaluation is meaningless.
train_row_idx = train_ratings["userId"].map(user_to_idx).values
train_col_idx = train_ratings["movieId_tmdb"].map(movie_to_idx).values
train_data = train_ratings["rating"].values

train_matrix = csr_matrix((train_data, (train_row_idx, train_col_idx)),
                           shape=(len(user_ids), len(movie_ids)))

# Refit SVD on the train-only matrix
svd_eval = TruncatedSVD(n_components=n_components, random_state=42)
train_user_factors = svd_eval.fit_transform(train_matrix)

In [ ]:
# For each user, get their top-K predicted movies (excluding ones already rated in train)
precisions = []
predicted_scores = train_user_factors @ svd_eval.components_
test_users = test_ratings["userId"].unique()
for u in test_users:
    if u not in user_to_idx:
        continue
    u_idx = user_to_idx[u]

    # Movies this user already rated in TRAIN — exclude these, we want NEW recommendations
    already_rated_train = set(train_ratings[train_ratings["userId"] == u]["movieId_tmdb"])
    already_rated_idx = {movie_to_idx[m] for m in already_rated_train if m in movie_to_idx}

    # Top-K predicted movies for this user, skipping ones already rated in train
    user_scores = predicted_scores[u_idx].copy()
    for idx in already_rated_idx:
        user_scores[idx] = -np.inf
    top_k_idx = np.argsort(user_scores)[::-1][:K]
    top_k_movie_ids = {movie_ids[i] for i in top_k_idx}

    # What did this user ACTUALLY like in the held-out test set?
    actually_liked_test = set(
        test_ratings[(test_ratings["userId"] == u) & (test_ratings["rating"] >= LIKE_THRESHOLD)]["movieId_tmdb"]
    )

    if len(actually_liked_test) == 0:
        continue  # can't evaluate precision for a user with no liked test items

    hits = len(top_k_movie_ids & actually_liked_test)
    precision_at_k = hits / K
    precisions.append(precision_at_k)

In [ ]:
mean_precision = np.mean(precisions)
print(f"\nEvaluated on {len(precisions)} users")
print(f"Mean precision@{K}: {mean_precision:.4f}")
print(f"(i.e. on average, {mean_precision*K:.2f} out of every {K} recommended movies "
      f"were ones the user actually rated >= {LIKE_THRESHOLD} in held-out data)")


Evaluated on 648 users
Mean precision@10: 0.2108
(i.e. on average, 2.11 out of every 10 recommended movies were ones the user actually rated >= 4.0 in held-out data)
